# Part 3 Basic Machine Learning Walkthrough
This notebook uses simple Python steps. Run each cell from top to bottom. The full project can also be run with `python basic_train.py`.

## 1 Load the processed data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_folder = Path.cwd().parent
data = pd.read_csv(project_folder / 'data' / 'traffic_processed.csv')
data.head()

## 2 Create the proxy risk label
No accident dataset was provided. High risk means High or Severe congestion together with severe or low-visibility weather.

In [ ]:
high_congestion = data['congestion_category'].isin(['High', 'Severe'])
risky_weather = (data['is_severe_weather'] == 1) | (data['is_low_visibility'] == 1)
data['high_risk'] = (high_congestion & risky_weather).astype(int)
data['high_risk'].value_counts()

## 3 Select simple features and split the data

In [ ]:
features = [
    'temp_celsius', 'rain_1h', 'snow_1h', 'clouds_all',
    'is_holiday', 'is_weekend', 'is_low_visibility', 'is_severe_weather',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]

X = data[features]
split_row = int(len(data) * 0.80)
X_train = X.iloc[:split_row]
X_test = X.iloc[split_row:]

## 4 Classification models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y = data['high_risk']
y_train = y.iloc[:split_row]
y_test = y.iloc[split_row:]

logistic_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
forest_model = RandomForestClassifier(n_estimators=80, max_depth=12, class_weight='balanced', random_state=42)

logistic_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

for name, model in [('Logistic Regression', logistic_model), ('Random Forest', forest_model)]:
    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    print(name)
    print('Accuracy:', accuracy_score(y_test, prediction))
    print('Precision:', precision_score(y_test, prediction, zero_division=0))
    print('Recall:', recall_score(y_test, prediction, zero_division=0))
    print('F1:', f1_score(y_test, prediction, zero_division=0))
    print('ROC AUC:', roc_auc_score(y_test, probability))

## 5 Regression models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

traffic = data['traffic_volume']
traffic_train = traffic.iloc[:split_row]
traffic_test = traffic.iloc[split_row:]

linear_model = LinearRegression()
regression_forest = RandomForestRegressor(n_estimators=80, max_depth=14, random_state=42)

linear_model.fit(X_train, traffic_train)
regression_forest.fit(X_train, traffic_train)

for name, model in [('Linear Regression', linear_model), ('Random Forest', regression_forest)]:
    prediction = model.predict(X_test)
    print(name, 'MAE:', mean_absolute_error(traffic_test, prediction), 'R2:', r2_score(traffic_test, prediction))

## 6 K-means clustering

In [ ]:
from sklearn.cluster import KMeans

cluster_columns = ['hour_sin', 'hour_cos', 'traffic_volume', 'is_severe_weather']
cluster_values = StandardScaler().fit_transform(data[cluster_columns])
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
data['cluster'] = kmeans.fit_predict(cluster_values)
data.groupby('cluster')[['hour', 'traffic_volume', 'is_severe_weather']].mean()

## 7 Basic association rules
The full `basic_train.py` file calculates support, confidence and lift for time-period to congestion rules.

In [ ]:
rules = pd.read_csv(project_folder / 'models' / 'association_rules.csv')
rules.head(10)

## 8 Basic neural network

In [ ]:
from sklearn.neural_network import MLPRegressor

neural_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=200, early_stopping=True, tol=0.01, random_state=42)
)
neural_model.fit(X_train, traffic_train)
neural_prediction = neural_model.predict(X_test)
print('Neural network MAE:', mean_absolute_error(traffic_test, neural_prediction))
print('Neural network R2:', r2_score(traffic_test, neural_prediction))

## 9 LIME explanation
LIME explains one prediction by testing small changes around that record.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    X_train.values,
    feature_names=features,
    mode='regression',
    random_state=42
)

def predict_for_lime(values):
    table = pd.DataFrame(values, columns=features)
    return neural_model.predict(table)

explanation = explainer.explain_instance(
    X_test.iloc[0].values,
    predict_for_lime,
    num_features=8
)
explanation.as_list()

## 10 Next steps
After running this notebook, review the `mlflow`, `deployment`, `recommendation_system`, and `monitoring` folders. Each folder contains one small Python example.